In [5]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# Config (LOCAL PATHS)
# =========================
DATASETS = {
    "NOTEARS": "./training_data_normalized_with_edge_features_NOTEARS.csv",
    "PC":      "./training_data_normalized_with_edge_features_PC.csv",
    "GES":     "./training_data_normalized_with_edge_features_GES.csv",
    "GOLEM":   "./training_data_normalized_with_edge_features_GOLEM.csv",
}

OUT_DIR = "./eda_png_only"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------- Distribution PNG settings ----------
# 한 장에 몇 개 feature를 넣을지 (3x4=12개 추천)
GRID_NROWS = 3
GRID_NCOLS = 4
PER_PAGE = GRID_NROWS * GRID_NCOLS

# edge 분포 전체를 “페이지 묶음 PNG”로 출력
EDGE_HIST_BINS = 60
EDGE_PLOT_MAX_PAGES = None  # None이면 전체 페이지 다 그림. 너무 많으면 정수로 제한 (예: 20)

# 원본 feature도 보고 싶으면 True
PLOT_ORIGINAL_FEATURES = False
ORIG_HIST_BINS = 60
ORIG_PLOT_MAX_PAGES = None

# boxplot도 같이 만들지 여부
MAKE_BOXPLOTS = True
BOX_MAX_PAGES = None  # None=전체

# ---------- Correlation Heatmap settings ----------
# 히트맵은 DAG별 1개씩만 만들고, 셀 안에 수치 표시(annotate)
# 컬럼이 너무 많으면 annotate가 사실상 불가능하므로,
# "원본 전부 + edge 상위 TOPK_EDGES_FOR_HEATMAP"로 히트맵을 구성
TOPK_EDGES_FOR_HEATMAP = 60   # edge가 많을 때 히트맵에 포함할 edge 수(분산 큰 순)
HEATMAP_METHOD = "pearson"    # pearson 권장
HEATMAP_FIGSIZE_BASE = 0.6    # 컬럼당 크기 스케일(너무 크면 조절)
HEATMAP_ANNOTATE = True       # 셀 내부 수치 표시
HEATMAP_DECIMALS = 2          # 표시 소수 자릿수
HEATMAP_MAX_COLS_HARD = 140   # 안전장치: 이 이상이면 강제로 줄임(annotate 목적)


# =========================
# Helpers
# =========================
def detect_target_col(df: pd.DataFrame) -> str:
    candidates = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError("Target column not found among common candidates.")

def split_cols(df: pd.DataFrame, target_col: str):
    edge_cols = [c for c in df.columns if c.startswith("edge_")]
    orig_cols = [c for c in df.columns if (c != target_col and not c.startswith("edge_"))]
    return orig_cols, edge_cols

def to_numeric_df(df: pd.DataFrame, cols):
    out = df[cols].copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def save_fig(fig, name: str, dpi: int = 200):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"[SAVED] {path}")

def chunk_list(xs, size):
    for i in range(0, len(xs), size):
        yield xs[i:i+size]

def hist_grid(df_num: pd.DataFrame, cols: list, bins: int, title: str, out_png: str):
    fig = plt.figure(figsize=(4 * GRID_NCOLS, 3 * GRID_NROWS))
    for i, c in enumerate(cols):
        ax = plt.subplot(GRID_NROWS, GRID_NCOLS, i + 1)
        x = df_num[c].to_numpy(dtype=np.float64)
        x = x[~np.isnan(x)]
        if x.size == 0:
            ax.set_title(c[:40], fontsize=8)
            ax.text(0.5, 0.5, "empty", ha="center", va="center")
            ax.set_axis_off()
            continue

        ax.hist(x, bins=bins)
        mu = float(np.mean(x))
        sd = float(np.std(x))
        ax.set_title(c[:40], fontsize=8)
        ax.text(0.02, 0.95, f"mean={mu:.3f}\nstd={sd:.3f}",
                transform=ax.transAxes, va="top", fontsize=8)

    plt.suptitle(title)
    plt.tight_layout()
    save_fig(fig, out_png)

def box_grid(df_num: pd.DataFrame, cols: list, title: str, out_png: str):
    fig = plt.figure(figsize=(4 * GRID_NCOLS, 3 * GRID_NROWS))
    for i, c in enumerate(cols):
        ax = plt.subplot(GRID_NROWS, GRID_NCOLS, i + 1)
        x = df_num[c].to_numpy(dtype=np.float64)
        x = x[~np.isnan(x)]
        if x.size == 0:
            ax.set_title(c[:40], fontsize=8)
            ax.text(0.5, 0.5, "empty", ha="center", va="center")
            ax.set_axis_off()
            continue

        ax.boxplot(x, vert=True, showfliers=True)
        mu = float(np.mean(x))
        sd = float(np.std(x))
        ax.set_title(c[:40], fontsize=8)
        ax.text(0.02, 0.95, f"mean={mu:.3f}\nstd={sd:.3f}",
                transform=ax.transAxes, va="top", fontsize=8)

    plt.suptitle(title)
    plt.tight_layout()
    save_fig(fig, out_png)

def corr_heatmap_annotated(corr: pd.DataFrame, title: str, out_png: str):
    cols = list(corr.columns)
    n = len(cols)

    # figsize: 컬럼 수에 비례 (너무 커지면 조절)
    size = max(10, n * HEATMAP_FIGSIZE_BASE)
    fig = plt.figure(figsize=(size, size))

    mat = corr.to_numpy()
    plt.imshow(mat, aspect="auto", interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    plt.xticks(range(n), cols, rotation=90, fontsize=8)
    plt.yticks(range(n), cols, fontsize=8)

    if HEATMAP_ANNOTATE:
        # 수치 오버레이
        for i in range(n):
            for j in range(n):
                v = mat[i, j]
                if np.isnan(v):
                    txt = "nan"
                else:
                    txt = f"{v:.{HEATMAP_DECIMALS}f}"
                # 배경이 진할 때 글자가 안 보일 수 있으니 단순 규칙으로 글자색 결정
                color = "white" if (not np.isnan(v) and abs(v) > 0.5) else "black"
                plt.text(j, i, txt, ha="center", va="center", fontsize=7, color=color)

    plt.tight_layout()
    save_fig(fig, out_png)


# =========================
# Main
# =========================
def main():
    # Load & detect target
    dfs = {}
    target_col = None

    for alg, path in DATASETS.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing file: {path}")
        df = pd.read_csv(path, low_memory=False)
        dfs[alg] = df
        if target_col is None:
            target_col = detect_target_col(df)

    print(f"[INFO] target_col = {target_col}")
    print(f"[INFO] outputs -> {OUT_DIR}")

    # 1) Distribution PNGs (orig + edge)
    for alg, df in dfs.items():
        print(f"\n===== {alg} =====")
        orig_cols, edge_cols = split_cols(df, target_col)
        print(f"[INFO] rows={df.shape[0]} orig={len(orig_cols)} edge={len(edge_cols)}")

        # Original feature distributions
        if PLOT_ORIGINAL_FEATURES and len(orig_cols) > 0:
            Xo = to_numeric_df(df, orig_cols)
            pages = list(chunk_list(orig_cols, PER_PAGE))
            if ORIG_PLOT_MAX_PAGES is not None:
                pages = pages[:ORIG_PLOT_MAX_PAGES]

            for p, cols_chunk in enumerate(pages, start=1):
                hist_grid(
                    Xo, cols_chunk, bins=ORIG_HIST_BINS,
                    title=f"{alg} - Original feature hist (page {p}/{len(pages)})",
                    out_png=f"{alg}_orig_hist_p{p:03d}.png",
                )
                if MAKE_BOXPLOTS:
                    if (BOX_MAX_PAGES is None) or (p <= BOX_MAX_PAGES):
                        box_grid(
                            Xo, cols_chunk,
                            title=f"{alg} - Original feature box (page {p}/{len(pages)})",
                            out_png=f"{alg}_orig_box_p{p:03d}.png",
                        )

        # Edge feature distributions (전체 edge를 페이지별 PNG로)
        if len(edge_cols) > 0:
            Xe = to_numeric_df(df, edge_cols)
            pages = list(chunk_list(edge_cols, PER_PAGE))
            if EDGE_PLOT_MAX_PAGES is not None:
                pages = pages[:EDGE_PLOT_MAX_PAGES]

            for p, cols_chunk in enumerate(pages, start=1):
                hist_grid(
                    Xe, cols_chunk, bins=EDGE_HIST_BINS,
                    title=f"{alg} - Edge feature hist (page {p}/{len(pages)})",
                    out_png=f"{alg}_edge_hist_p{p:03d}.png",
                )
                if MAKE_BOXPLOTS:
                    if (BOX_MAX_PAGES is None) or (p <= BOX_MAX_PAGES):
                        box_grid(
                            Xe, cols_chunk,
                            title=f"{alg} - Edge feature box (page {p}/{len(pages)})",
                            out_png=f"{alg}_edge_box_p{p:03d}.png",
                        )

    # 2) Correlation Heatmaps (총 4개: DAG별 1개)
    # 요구사항: data별(=DAG별) 원본+edge 포함 히트맵 1개씩
    for alg, df in dfs.items():
        orig_cols, edge_cols = split_cols(df, target_col)

        # 히트맵 구성: 원본은 전부 + edge는 top-k(std 큰 순)
        cols = list(orig_cols)

        if len(edge_cols) > 0:
            Xe = to_numeric_df(df, edge_cols)
            edge_std = Xe.std(skipna=True).sort_values(ascending=False)
            top_edges = edge_std.head(TOPK_EDGES_FOR_HEATMAP).index.tolist()
            cols += top_edges

        # 안전장치: 너무 많으면 추가로 줄임
        if len(cols) > HEATMAP_MAX_COLS_HARD:
            # 원본은 유지하고, edge만 더 줄임
            keep_orig = orig_cols
            remain = HEATMAP_MAX_COLS_HARD - len(keep_orig)
            remain = max(0, remain)
            cols = keep_orig + cols[len(keep_orig):len(keep_orig)+remain]

        X = to_numeric_df(df, cols)
        corr = X.corr(method=HEATMAP_METHOD)

        corr_heatmap_annotated(
            corr,
            title=f"{alg} - Corr heatmap (orig + top{min(TOPK_EDGES_FOR_HEATMAP, len(edge_cols))} edges by std)",
            out_png=f"{alg}_corr_heatmap_annotated.png",
        )

    print("\n[DONE] PNG-only EDA complete.")


if __name__ == "__main__":
    main()


[INFO] target_col = label
[INFO] outputs -> ./eda_png_only

===== NOTEARS =====
[INFO] rows=17881 orig=14 edge=9
[SAVED] ./eda_png_only\NOTEARS_edge_hist_p001.png
[SAVED] ./eda_png_only\NOTEARS_edge_box_p001.png

===== PC =====
[INFO] rows=17881 orig=14 edge=24
[SAVED] ./eda_png_only\PC_edge_hist_p001.png
[SAVED] ./eda_png_only\PC_edge_box_p001.png
[SAVED] ./eda_png_only\PC_edge_hist_p002.png
[SAVED] ./eda_png_only\PC_edge_box_p002.png

===== GES =====
[INFO] rows=17881 orig=14 edge=55
[SAVED] ./eda_png_only\GES_edge_hist_p001.png
[SAVED] ./eda_png_only\GES_edge_box_p001.png
[SAVED] ./eda_png_only\GES_edge_hist_p002.png
[SAVED] ./eda_png_only\GES_edge_box_p002.png
[SAVED] ./eda_png_only\GES_edge_hist_p003.png
[SAVED] ./eda_png_only\GES_edge_box_p003.png
[SAVED] ./eda_png_only\GES_edge_hist_p004.png
[SAVED] ./eda_png_only\GES_edge_box_p004.png
[SAVED] ./eda_png_only\GES_edge_hist_p005.png
[SAVED] ./eda_png_only\GES_edge_box_p005.png

===== GOLEM =====
[INFO] rows=17881 orig=14 edge=11
[